# Paper 2 — Three-Arm Embedding Divergence Control

**Runtime: CPU, minutes. Downloads a sentence-transformer model (needs internet).**

Compares embedding-space trajectory divergence across three arms — REVISION, matched FILLER, and RANDOM positions. The three-arm test uses **no scope labels at all**: arms are defined by revision status and position, so this result survives the scope cut intact.

Only the optional cross-check against the scope-labeled event set is gated behind `SCOPE_ANALYSIS_ENABLED`.

Prerequisites on Drive: `labels/`, `rejects/`, `traces/`.


In [ ]:
# ---- Google Drive -----------------------------------------------------------
from google.colab import drive
drive.mount("/content/gdrive")

In [ ]:
# ---- scope_flag (inlined; no file to place) ---------------------------------
# The analysis scripts below contain `import scope_flag` and gate their retired
# local/global SCOPE sections on it. Rather than editing that logic -- which
# would risk changing behaviour -- this cell installs a real module object of
# that name into sys.modules, so the imports below resolve to it unchanged.
#
# SET THIS ONE TOGGLE:
SCOPE_ANALYSIS_ENABLED = False   # False = paper path (default). True = also
                                 # run the retired scope analysis.
#
# NOTE this governs scope ANALYSIS AND REPORTING only, never sample
# construction. The event sets were drawn stratified on the human
# local/global labels and no flag can change that. See SCOPE_ISOLATION.md §3.

import sys, types

_sf = types.ModuleType("scope_flag")
_sf.SCOPE_ANALYSIS_ENABLED = SCOPE_ANALYSIS_ENABLED
_sf.SCOPE_BANNER_SHOWN = False

def _scope_banner():
    if _sf.SCOPE_BANNER_SHOWN:
        return _sf.SCOPE_ANALYSIS_ENABLED
    _sf.SCOPE_BANNER_SHOWN = True
    if _sf.SCOPE_ANALYSIS_ENABLED:
        print("\n[scope_flag] SCOPE_ANALYSIS_ENABLED=True — exploratory scope "
              "sections WILL run.")
        print("[scope_flag] These are retired/exploratory and are not reported "
              "in the paper.")
    else:
        print("\n[scope_flag] SCOPE_ANALYSIS_ENABLED=False — scope sections "
              "skipped (paper path).")
        print("[scope_flag] Set SCOPE_ANALYSIS_ENABLED=True above to run them.")
    return _sf.SCOPE_ANALYSIS_ENABLED

def _skip_note(section_name):
    print(f"\n[scope_flag] SKIPPED (retired scope analysis): {section_name}")
    print("[scope_flag]   re-enable by setting SCOPE_ANALYSIS_ENABLED = True above")

_sf.scope_banner = _scope_banner
_sf.skip_note = _skip_note
sys.modules["scope_flag"] = _sf
print(f"scope_flag installed. SCOPE_ANALYSIS_ENABLED = {SCOPE_ANALYSIS_ENABLED}")

## Three-arm divergence control


In [ ]:
# =============================================================================
# EMBEDDING DIVERGENCE CONTROL
#
# THE THREAT
# Trajectory divergence = cosine distance between the centroid of the sentences
# before a revision and the centroid of the sentences after it. But reasoning
# text drifts semantically as it proceeds REGARDLESS of revisions. If divergence
# at an arbitrary point in a trace is the same as at a revision, then divergence
# is measuring generic semantic drift, not anything about revisions -- exactly
# the failure mode the filler control exposed for causal necessity.
#
# THREE ARMS, identical extraction and metric:
#   REVISION - confirmed revision positions (labels/*.json, verified=true)
#   FILLER   - GPT-4o-verified NON-revision positions with the SAME marker words
#              (rejects/*_confirmed_filler.json), matched 1:1 on (model, marker).
#              This is the direct analogue of the necessity filler control.
#   RANDOM   - arbitrary positions in the same traces, no marker at all.
#              Establishes the baseline drift rate for text with no discourse
#              marker present, which the filler arm cannot isolate (fillers
#              still contain a marker word).
#
# INTERPRETATION
#   revision > filler ~= random  -> divergence responds to genuine revisions.
#   revision ~= filler > random  -> divergence responds to MARKER WORDS, not
#                                   revision status. Would mean the scope
#                                   result rests on a lexical artifact.
#   all three equal              -> divergence is generic drift. The scope
#                                   result would have to be withdrawn, as the
#                                   necessity finding was.
#
# WHY REBUILD THE REVISION ARM
# The original divergences came from annotation-workbook context; controls must
# come from raw traces. Comparing those directly would confound extraction
# differences with the effect. So all three arms are rebuilt here from raw
# traces with identical logic, replicating the workbook's own windowing
# (900/700 char budgets, then 6-sentence caps). The script cross-checks the
# rebuilt revision arm against the original kill-screen values.
#
# COST: CPU + internet for the embedding model. No GPU, minutes not hours.
# =============================================================================



from pathlib import Path
from collections import defaultdict, Counter
import json, random, re

import numpy as np
from scipy.stats import mannwhitneyu
from sentence_transformers import SentenceTransformer

In [ ]:
import scope_flag   # SCOPE_ANALYSIS_ENABLED gate; see SCOPE_ISOLATION.md

GDRIVE_BASE = Path("/content/gdrive/MyDrive/llm_planning_study")
LABELS_DIR  = GDRIVE_BASE / "labels"
TRACES_DIR  = GDRIVE_BASE / "traces"
REJECT_DIR  = GDRIVE_BASE / "rejects"
EXPORT_DIR  = GDRIVE_BASE / "hand_labeling_export"

In [ ]:
KILL_EVENTS_PATH = EXPORT_DIR / "kill_screen_events.json"   # for the cross-check
OUT_PATH         = EXPORT_DIR / "divergence_control.json"

EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
SEED = 8080

# replicate the annotation-workbook windowing exactly
CHAR_BUDGET_BEFORE = 900
CHAR_BUDGET_AFTER  = 700

In [ ]:
RAW_BUFFER_BEFORE  = 2500
RAW_BUFFER_AFTER   = 1200
N_BEFORE = N_AFTER = 6      # then the kill-screen sentence caps

MODELS = ["DeepSeek-R1-Distill-Llama-8B", "DeepSeek-R1-Distill-Qwen-1.5B",
          "DeepSeek-R1-Distill-Qwen-7B",  "DeepSeek-R1-Distill-Qwen-14B"]


# ---------------------------------------------------------------- helpers ---
def clean_bpe(text):
    return text.replace("\u0120", " ").replace("\u010a", "\n")


def split_sentences(text):
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    return [p for p in parts if p.strip()]

In [ ]:
def take_by_budget(sentences, budget, from_end):
    if not sentences:
        return []
    ordered = list(reversed(sentences)) if from_end else sentences
    out, total = [], 0
    for s in ordered:
        out.append(s); total += len(s) + 1
        if total >= budget:
            break
    return list(reversed(out)) if from_end else out


def normalize_marker(text):
    return re.sub(r"[^\w\s]", "", str(text).lower()).strip()


def extract_window(full_response, char_start, char_end):
    """Replicates build_blind_sample_xlsx.py's windowing, then the kill-screen
    6-sentence caps. Identical for all three arms."""
    raw_start = max(0, char_start - RAW_BUFFER_BEFORE)
    raw_end   = min(len(full_response), char_end + RAW_BUFFER_AFTER)
    window = clean_bpe(full_response[raw_start:raw_end])

    local_start = char_start - raw_start
    local_end   = char_end - raw_start
    if not (0 <= local_start <= local_end <= len(window)):
        return None, None

    before_txt = window[:local_start]
    after_txt  = window[local_end:]

    before = take_by_budget(split_sentences(before_txt), CHAR_BUDGET_BEFORE, from_end=True)
    after  = take_by_budget(split_sentences(after_txt),  CHAR_BUDGET_AFTER,  from_end=False)
    # drop a trailing fragment cut off by the raw buffer
    if after and not re.search(r'[.!?]\s*$', after[-1]):
        after = after[:-1]

    before = before[-N_BEFORE:]
    after  = after[:N_AFTER]
    if not before or not after:
        return None, None
    return before, after


def load_trace(model, pid, cache):
    key = (model, pid)
    if key not in cache:
        p = TRACES_DIR / model / f"{pid}.json"
        cache[key] = json.load(open(p)) if p.exists() else None
    return cache[key]


# ---------------------------------------------------------- build the arms ---
def build_revision_arm(cache):
    out = []
    for model in MODELS:
        mdir = LABELS_DIR / model
        if not mdir.exists():
            continue
        for f in mdir.glob("*.json"):
            obj = json.load(open(f))
            pid = obj.get("problem_id", f.stem)
            revs = [r for r in obj.get("revisions", []) or [] if r.get("verified", True)]
            if not revs:
                continue
            trace = load_trace(model, pid, cache)
            if trace is None or not trace.get("correct", False) or trace.get("truncated", False):
                continue
            fr = trace.get("full_response", "")
            for r in revs:
                cs, ce = r.get("char_start"), r.get("char_end")
                if cs is None or ce is None or not (0 <= cs < len(fr)):
                    continue
                b, a = extract_window(fr, cs, ce)
                if b is None:
                    continue
                out.append({"arm": "revision", "model": model, "problem_id": pid,
                            "marker": normalize_marker(r.get("matched_text", "")),
                            "before": b, "after": a})
    return out


def build_filler_arm(cache):
    out = []
    for model in MODELS:
        rp = REJECT_DIR / f"{model}_confirmed_filler.json"
        if not rp.exists():
            continue
        for flr in json.load(open(rp)):
            pid = flr["problem_id"]
            trace = load_trace(model, pid, cache)
            if trace is None or not trace.get("correct", False) or trace.get("truncated", False):
                continue
            fr = trace.get("full_response", "")
            cs = flr.get("char_start")
            matched = flr.get("matched_text", "")
            if cs is None or not (0 <= cs < len(fr)):
                continue
            ce = cs + len(matched)
            b, a = extract_window(fr, cs, min(ce, len(fr)))
            if b is None:
                continue
            out.append({"arm": "filler", "model": model, "problem_id": pid,
                        "marker": normalize_marker(matched),
                        "before": b, "after": a})
    return out

In [ ]:
def build_random_arm(cache, per_model_counts, rng):
    """Arbitrary positions in the same traces -- no marker present. Zero-width
    'event' so before/after split at a plain character offset."""
    out = []
    for model, n_wanted in per_model_counts.items():
        pids = [pid for (m, pid), tr in cache.items()
                if m == model and tr is not None
                and tr.get("correct", False) and not tr.get("truncated", False)]
        if not pids:
            continue
        attempts, made = 0, 0
        while made < n_wanted and attempts < n_wanted * 20:
            attempts += 1
            pid = pids[rng.randrange(len(pids))]
            fr = cache[(model, pid)].get("full_response", "")
            if len(fr) < 2000:
                continue
            cs = rng.randrange(1000, len(fr) - 800)
            b, a = extract_window(fr, cs, cs)   # zero-width split point
            if b is None:
                continue
            out.append({"arm": "random", "model": model, "problem_id": pid,
                        "marker": "", "before": b, "after": a})
            made += 1
    return out


# ------------------------------------------------------------------- stats ---
def describe(vals, label):
    v = np.array(vals)
    print(f"\n  {label}  (n={len(v)})")
    print(f"    mean {v.mean():.4f}   median {np.median(v):.4f}   sd {v.std():.4f}")
    print(f"    range [{v.min():.4f}, {v.max():.4f}]")
    return v


def compare(a, b, la, lb):
    u, p = mannwhitneyu(a, b, alternative="two-sided")
    auc = u / (len(a) * len(b))
    rng = np.random.RandomState(0)
    diffs = [np.median(rng.choice(a, len(a), True)) - np.median(rng.choice(b, len(b), True))
             for _ in range(5000)]
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    print(f"\n  {la} vs {lb}:")
    print(f"    Mann-Whitney p = {p:.4f}   P({la} > {lb}) = {auc:.3f}")
    print(f"    median difference = {np.median(a)-np.median(b):+.4f}   95% CI [{lo:+.4f}, {hi:+.4f}]")
    return {"p": float(p), "auc": float(auc),
            "median_diff": float(np.median(a) - np.median(b)), "ci95": [float(lo), float(hi)]}


def main():
    cache = {}
    print("Building revision arm...")
    rev = build_revision_arm(cache)
    print(f"  {len(rev)} revision events")

    print("Building filler arm...")
    flr_pool = build_filler_arm(cache)
    print(f"  {len(flr_pool)} eligible filler positions")

    # match filler 1:1 to revision on (model, marker)
    rng = random.Random(SEED)
    by_cell = defaultdict(list)
    for e in flr_pool:
        by_cell[(e["model"], e["marker"])].append(e)
    demand = Counter((e["model"], e["marker"]) for e in rev)
    flr, unmatched = [], 0
    for cell, n_needed in demand.items():
        avail = by_cell.get(cell, [])
        take = min(n_needed, len(avail))
        if take < n_needed:
            unmatched += n_needed - take
        if take:
            flr.extend(rng.sample(avail, take))
    print(f"  matched {len(flr)} filler controls ({unmatched} revision events unmatched)")

    print("Building random arm...")
    rnd = build_random_arm(cache, Counter(e["model"] for e in rev), rng)
    print(f"  {len(rnd)} random positions")

    # ---------------------------------------------------------- embed ------
    print(f"\nLoading {EMBED_MODEL_NAME} ...")
    embedder = SentenceTransformer(EMBED_MODEL_NAME)

    def divergence(ev):
        b = embedder.encode(ev["before"], normalize_embeddings=True)
        a = embedder.encode(ev["after"],  normalize_embeddings=True)
        cb, ca = b.mean(axis=0), a.mean(axis=0)
        return 1.0 - float(np.dot(cb, ca) / (np.linalg.norm(cb) * np.linalg.norm(ca)))

    print("Computing divergence for all arms...")
    for arm in (rev, flr, rnd):
        for ev in arm:
            ev["divergence"] = divergence(ev)

    # ------------------------------------------- cross-check vs original ---
    # The THREE-ARM TEST ITSELF uses no scope labels -- revision/filler/random
    # arms are defined by revision status and position, never by local/global.
    # The three-arm result is therefore KEPT and survives the scope cut intact.
    # Only this optional cross-check reads the scope-labeled event set, so it
    # is gated. Its input now lives in scope_analysis/kill_screen_events.json.
    if scope_flag.scope_banner() and KILL_EVENTS_PATH.exists():
        orig = json.load(open(KILL_EVENTS_PATH))
        orig_div = [divergence({"before": e["before_sents"], "after": e["after_sents"]})
                    for e in orig]
        rev_div_all = [e["divergence"] for e in rev]
        print(f"\n{'='*70}\nCROSS-CHECK: rebuilt revision arm vs. original kill-screen events\n{'='*70}")
        print(f"  original kill-screen events (n={len(orig_div)}): "
              f"median {np.median(orig_div):.4f}, mean {np.mean(orig_div):.4f}")
        print(f"  rebuilt revision arm (n={len(rev_div_all)}): "
              f"median {np.median(rev_div_all):.4f}, mean {np.mean(rev_div_all):.4f}")
        print("  (These use different event sets -- the rebuilt arm covers ALL confirmed")
        print("   revisions, the original only the human-labeled subset -- so expect")
        print("   similar but not identical distributions. A large gap would indicate the")
        print("   windowing replication is wrong and the comparison below is unsafe.)")

    # ------------------------------------------------------------ results --
    print(f"\n{'='*70}\nDIVERGENCE BY ARM\n{'='*70}")
    v_rev = describe([e["divergence"] for e in rev], "REVISION positions")
    v_flr = describe([e["divergence"] for e in flr], "FILLER positions (same markers, verified non-revision)")
    v_rnd = describe([e["divergence"] for e in rnd], "RANDOM positions (no marker)")

    print(f"\n{'='*70}\nARM COMPARISONS\n{'='*70}")
    res = {}
    res["revision_vs_filler"] = compare(v_rev, v_flr, "revision", "filler")
    res["revision_vs_random"] = compare(v_rev, v_rnd, "revision", "random")
    res["filler_vs_random"]   = compare(v_flr, v_rnd, "filler",   "random")

    print(f"\n  per-model medians:")
    for m in MODELS:
        r_ = [e["divergence"] for e in rev if e["model"] == m]
        f_ = [e["divergence"] for e in flr if e["model"] == m]
        d_ = [e["divergence"] for e in rnd if e["model"] == m]
        fmt = lambda x: f"{np.median(x):.4f}" if x else "  -   "
        print(f"    {m:<32} rev {fmt(r_)} (n={len(r_):>4})  "
              f"fil {fmt(f_)} (n={len(f_):>4})  rnd {fmt(d_)} (n={len(d_):>4})")

    # --------------------------------------------------------------- read --
    print(f"\n{'#'*70}\n# READ\n{'#'*70}")
    p_rf = res["revision_vs_filler"]["p"]
    p_rr = res["revision_vs_random"]["p"]
    p_fr = res["filler_vs_random"]["p"]
    ci_rr = res["revision_vs_random"]["ci95"]
    med_rev, med_flr, med_rnd = np.median(v_rev), np.median(v_flr), np.median(v_rnd)

    beats_filler = p_rf < 0.05 and med_rev > med_flr
    beats_random = p_rr < 0.05 and med_rev > med_rnd

    # RANDOM is the decisive control. Filler positions still contain a marker
    # word AND are, by GPT-4o's own definition, points where no genuine change
    # occurs -- so they may sit at unusually CONTINUOUS points in the reasoning,
    # making them an artificially low baseline rather than a neutral one.
    # Beating filler alone therefore does NOT establish revision-sensitivity;
    # beating arbitrary text does. Both must hold.
    if beats_filler and beats_random:
        print("Divergence exceeds BOTH controls -- matched fillers and random positions.")
        print("It responds to revision status rather than to marker words or generic")
        print("drift -- i.e. divergence is a revision-sensitive measurement.")

    elif beats_filler and not beats_random:
        print("SPLIT RESULT -- divergence beats fillers but NOT random positions.")
        print(f"  revision vs filler : +{med_rev-med_flr:.4f}  (p={p_rf:.4g})")
        print(f"  revision vs random : {med_rev-med_rnd:+.4f}  (p={p_rr:.4g}, "
              f"95% CI [{ci_rr[0]:+.4f}, {ci_rr[1]:+.4f}])")
        print()
        print("Divergence at a revision is NOT distinguishable from divergence at an")
        print("arbitrary point in the same trace. It therefore does not identify")
        print("revisions against a generic-text baseline, and is not usable as evidence")
        print("that a semantic shift occurs at revisions.")
        print()
        print("The revision-filler gap is most likely explained by fillers being")
        print("unusually CONTINUOUS -- they are, by construction, marker words at points")
        print("where no genuine change occurs -- rather than by revisions being")
        print("unusually divergent. Note filler also sits BELOW random "
              f"({med_flr:.4f} vs {med_rnd:.4f}), which is consistent with that reading.")
        print()
        print("Divergence is not inert (it separates revisions from discourse padding,")
        print("robustly and in every model), but what it tracks is not 'a semantic shift")
        print("is happening here'. Report the random-arm null alongside any use of it.")

    elif not beats_filler and p_fr < 0.05 and med_flr > med_rnd:
        print("Divergence does NOT distinguish revisions from same-marker fillers, but")
        print("BOTH exceed random positions. This implicates the MARKER WORD, not")
        print("revision status -- divergence tracks the lexical/discourse environment")
        print("around these words rather than whether a genuine revision occurred.")
        print("Divergence would then be a lexical artifact -- a direct parallel to the")
        print("confound Task 1 was built to rule out.")

    elif p_rf >= 0.05 and p_rr >= 0.05:
        print("Divergence is indistinguishable across ALL THREE arms. It measures generic")
        print("semantic drift in reasoning text, not anything about revisions. Any claim")
        print("resting on embedding divergence must be withdrawn on the same grounds as")
        print("the causal-necessity finding.")

    else:
        print("Pattern not covered by the standard readings -- interpret the three")
        print("pairwise comparisons directly. Key question: does revision exceed RANDOM?")
        print(f"  revision vs random: {med_rev-med_rnd:+.4f}, p={p_rr:.4g}, "
              f"CI [{ci_rr[0]:+.4f}, {ci_rr[1]:+.4f}]")

    print("\nNote: RANDOM is the decisive control here, not FILLER. Filler positions")
    print("still contain a marker word and may be atypically continuous, so they are")
    print("not a neutral baseline. Check CI widths before calling any null conclusive.")

    json.dump({
        "n_revision": len(v_rev), "n_filler": len(v_flr), "n_random": len(v_rnd),
        "median_revision": float(np.median(v_rev)),
        "median_filler": float(np.median(v_flr)),
        "median_random": float(np.median(v_rnd)),
        "mean_revision": float(v_rev.mean()), "mean_filler": float(v_flr.mean()),
        "mean_random": float(v_rnd.mean()),
        "comparisons": res,
    }, open(OUT_PATH, "w"), indent=2)
    print(f"\nWrote {OUT_PATH}")


if __name__ == "__main__":
    main()